In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline 
from sklearn.compose import make_column_selector, ColumnTransformer
from sklearn.linear_model import LinearRegression 
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.model_selection import KFold, GridSearchCV
os.chdir("D:\Machine Learning\Datasets\Assignments")

In [2]:
train = pd.read_csv("train_v9rqX0R.csv")
test = pd.read_csv("test_AbJTz2l.csv")
print(train.columns)

Index(['Item_Identifier', 'Item_Weight', 'Item_Fat_Content', 'Item_Visibility',
       'Item_Type', 'Item_MRP', 'Outlet_Identifier',
       'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type',
       'Outlet_Type', 'Item_Outlet_Sales'],
      dtype='object')


In [3]:
train.shape, test.shape

((8523, 12), (5681, 11))

In [4]:
train.isnull().sum()

Item_Identifier                 0
Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

In [5]:
test.isnull().sum()

Item_Identifier                 0
Item_Weight                   976
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  1606
Outlet_Location_Type            0
Outlet_Type                     0
dtype: int64

In [6]:
items_trn = train[['Item_Identifier', 'Item_Weight']]
items_trn = items_trn.drop_duplicates()
items_tst = test[['Item_Identifier', 'Item_Weight']]
items_tst = items_tst.drop_duplicates()
items = pd.concat([items_trn, items_tst])
items = items.dropna()
items = items.drop_duplicates()
items.columns = ['Item_Identifier', 'I_Weight']
items.isnull().sum()

Item_Identifier    0
I_Weight           0
dtype: int64

In [7]:
train_1 = train.merge(items, how='left', on='Item_Identifier')
train_1 = train_1.drop('Item_Weight', axis=1)
train_1.isnull().sum()

Item_Identifier                 0
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
I_Weight                        0
dtype: int64

In [8]:
train_1.columns

Index(['Item_Identifier', 'Item_Fat_Content', 'Item_Visibility', 'Item_Type',
       'Item_MRP', 'Outlet_Identifier', 'Outlet_Establishment_Year',
       'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type',
       'Item_Outlet_Sales', 'I_Weight'],
      dtype='object')

In [9]:
outlets = train_1[['Outlet_Identifier', 'Outlet_Establishment_Year',
       'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type']]
outlets = outlets.drop_duplicates()
outlets

,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type
0,OUT049,1999,Medium,Tier 1,Supermarket Type1
1,OUT018,2009,Medium,Tier 3,Supermarket Type2
3,OUT010,1998,NaN,Tier 3,Grocery Store
4,OUT013,1987,High,Tier 3,Supermarket Type1
7,OUT027,1985,Medium,Tier 3,Supermarket Type3
8,OUT045,2002,NaN,Tier 2,Supermarket Type1
9,OUT017,2007,NaN,Tier 2,Supermarket Type1
11,OUT046,1997,Small,Tier 1,Supermarket Type1
19,OUT035,2004,Small,Tier 2,Supermarket Type1
23,OUT019,1985,Small,Tier 1,Grocery Store


In [10]:
outlets = outlets[['Outlet_Identifier','Outlet_Size']]
#outlets.to_csv("Outlets.csv", index=False)

In [11]:
outlets = outlets[['Outlet_Identifier','Outlet_Size']]
outlets = outlets.dropna()
outlets_imp = pd.DataFrame({'Outlet_Identifier':['OUT010','OUT017','OUT045'],
                            'Outlet_Size':["Small","High","High"]})
outlets = pd.concat([outlets,outlets_imp], ignore_index=True)
outlets.columns=['Outlet_Identifier','O_Size']
train_2 = train_1.merge(outlets, how='left', on='Outlet_Identifier')
train_2 = train_2.drop('Outlet_Size', axis=1)
train_2.shape


(8523, 12)

In [12]:
pd.crosstab( train_2['Outlet_Type'] , train_2['O_Size'] , margins=True)

O_Size,High,Medium,Small,All
Outlet_Type,,,,
Grocery Store,0,0,1083,1083
Supermarket Type1,2787,930,1860,5577
Supermarket Type2,0,928,0,928
Supermarket Type3,0,935,0,935
All,2787,2793,2943,8523


In [13]:
train_2.isnull().sum()

Item_Identifier              0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Location_Type         0
Outlet_Type                  0
Item_Outlet_Sales            0
I_Weight                     0
O_Size                       0
dtype: int64

OR

In [14]:
outlets = train_1[['Outlet_Identifier','Outlet_Size']]
outlets = outlets.drop_duplicates()
size_map = {
    "OUT010": "Small",
    "OUT017": "High",
    "OUT045": "High"
}

train_1["Outlet_Size"] = train_1["Outlet_Size"].fillna(
    train_1["Outlet_Identifier"].map(size_map)
)
# Validating the processing
pd.crosstab( train_1['Outlet_Type'] , train_1['Outlet_Size'] , margins=True)

Outlet_Size,High,Medium,Small,All
Outlet_Type,,,,
Grocery Store,0,0,1083,1083
Supermarket Type1,2787,930,1860,5577
Supermarket Type2,0,928,0,928
Supermarket Type3,0,935,0,935
All,2787,2793,2943,8523


In [15]:
train_1['Item_Fat_Content'].value_counts()

Item_Fat_Content
Low Fat    5089
Regular    2889
LF          316
reg         117
low fat     112
Name: count, dtype: int64

In [16]:
train_1['Item_Fat_Content'].replace({'reg':'Regular',
                                   'LF':'Low Fat',
                                   'low fat':'Low Fat'},
                                  inplace=True)
train_1['Item_Fat_Content'].value_counts()

C:\Users\dbda.STUDENTSDC\AppData\Local\Temp\ipykernel_17540\190472228.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_1['Item_Fat_Content'].replace({'reg':'Regular',


Item_Fat_Content
Low Fat    5517
Regular    3006
Name: count, dtype: int64

In [17]:
train_1.columns

Index(['Item_Identifier', 'Item_Fat_Content', 'Item_Visibility', 'Item_Type',
       'Item_MRP', 'Outlet_Identifier', 'Outlet_Establishment_Year',
       'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type',
       'Item_Outlet_Sales', 'I_Weight'],
      dtype='object')

In [18]:
train_1['Item_Type'].value_counts()

Item_Type
Fruits and Vegetables    1232
Snack Foods              1200
Household                 910
Frozen Foods              856
Dairy                     682
Canned                    649
Baking Goods              648
Health and Hygiene        520
Soft Drinks               445
Meat                      425
Breads                    251
Hard Drinks               214
Others                    169
Starchy Foods             148
Breakfast                 110
Seafood                    64
Name: count, dtype: int64

In [19]:
train_1['Outlet_Type'].value_counts()

Outlet_Type
Supermarket Type1    5577
Grocery Store        1083
Supermarket Type3     935
Supermarket Type2     928
Name: count, dtype: int64

In [94]:
train_1

,Item_Identifier,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,I_Weight
0,FDA15,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,9.300
1,DRC01,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,5.920
2,FDN15,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700,17.500
3,FDX07,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,Small,Tier 3,Grocery Store,732.3800,19.200
4,NCD19,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,8.930
...,...,...,...,...,...,...,...,...,...,...,...,...
8518,FDF22,Low Fat,0.056783,Snack Foods,214.5218,OUT013,1987,High,Tier 3,Supermarket Type1,2778.3834,6.865
8519,FDS36,Regular,0.046982,Baking Goods,108.1570,OUT045,2002,High,Tier 2,Supermarket Type1,549.2850,8.380
8520,NCJ29,Low Fat,0.035186,Health and Hygiene,85.1224,OUT035,2004,Small,Tier 2,Supermarket Type1,1193.1136,10.600
8521,FDN46,Regular,0.145221,Snack Foods,103.1332,OUT018,2009,Medium,Tier 3,Supermarket Type2,1845.5976,7.210


0

In [22]:
def clean_data(df):
    df_1 = df.merge(items, how='left', on='Item_Identifier')
    df_1 = df_1.drop('Item_Weight', axis=1)

    outlets = df_1[['Outlet_Identifier','Outlet_Size']]
    outlets = outlets.drop_duplicates()
    size_map = {
        "OUT010": "Small",
        "OUT017": "High",
        "OUT045": "High"
    }

    df_1["Outlet_Size"] = df_1["Outlet_Size"].fillna(
        df_1["Outlet_Identifier"].map(size_map)
    )

    df_1['Item_Fat_Content'] = df_1['Item_Fat_Content'].replace({'reg':'Regular',
                                   'LF':'Low Fat',
                                   'low fat':'Low Fat'})
    return df_1

train_1 = clean_data(train)
test_1 = clean_data(test)
# Validating the processing
#pd.crosstab( train_1['Outlet_Type'] , train_1['Outlet_Size'] , margins=True)
pd.crosstab( test_1['Outlet_Type'] , test_1['Outlet_Size'] , margins=True)

Outlet_Size,High,Medium,Small,All
Outlet_Type,,,,
Grocery Store,0,0,722,722
Supermarket Type1,1857,620,1240,3717
Supermarket Type2,0,618,0,618
Supermarket Type3,0,624,0,624
All,1857,1862,1962,5681


In [134]:
train_1.shape, test_1.shape

((8523, 12), (5681, 11))

In [135]:
train.shape, test.shape

((8523, 12), (5681, 11))

In [136]:
test_1.isnull().sum()

Item_Identifier              0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
I_Weight                     0
dtype: int64

In [20]:
train_1.columns

Index(['Item_Identifier', 'Item_Fat_Content', 'Item_Visibility', 'Item_Type',
       'Item_MRP', 'Outlet_Identifier', 'Outlet_Establishment_Year',
       'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type',
       'Item_Outlet_Sales', 'I_Weight'],
      dtype='object')

In [23]:
X_train = train_1.drop(['Item_Identifier',
                                 'Outlet_Identifier','Item_Outlet_Sales'], axis=1)
X_test = test_1.drop(['Item_Identifier', 'Outlet_Identifier'], axis=1)
y_train = train_1['Item_Outlet_Sales']
ohe = OneHotEncoder()
col_trnf = ColumnTransformer([('OHE',ohe, make_column_selector(dtype_include=object) )],
                             remainder='passthrough', verbose_feature_names_out=False)
rf = RandomForestRegressor(random_state=25)
pipe = Pipeline([('TRNS', col_trnf),('MODEL',rf)])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

In [139]:
ss = pd.read_csv("sample_submission_8RXa3c6.csv")
ss['Item_Outlet_Sales'] = y_pred 
ss.to_csv("ss_rf_20_dec_25.csv", index=False)

In [140]:
test.shape, ss.shape

((5681, 11), (5681, 3))

In [141]:
from lightgbm import LGBMRegressor

In [ ]:
l_gbm = LGBMRegressor(random_state=25, verbose=-1)
pipe = Pipeline([('TRNS', col_trnf),('MODEL',l_gbm)])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

In [146]:
ss = pd.read_csv("sample_submission_8RXa3c6.csv")
ss['Item_Outlet_Sales'] = y_pred 
ss.to_csv("ss_lgb_20_dec_25.csv", index=False)